# Clase 13 — Memorice

Vamos a construir el juego del memorice desde cero. La idea es que todo lo que vimos hasta ahora aparezca de forma natural: clases, encapsulación, colores en terminal y lógica.

---

## Diseño

Antes de escribir código, pensamos en qué objetos existen en el juego real:

- Una **ficha** tiene un valor, puede estar boca arriba o boca abajo, y puede estar encontrada o no.
- Un **tablero** organiza las fichas en una grilla y sabe cómo imprimirse.
- Un **juego** controla los turnos, pide input al jugador y decide si hay pareja.

```
Ficha       → qué soy, si estoy visible, si ya fui encontrada
Tablero     → dónde está cada ficha, cómo me imprimo con colores
Juego       → lógica de turnos, input del jugador, condición de victoria
```

El `Juego` contiene un `Tablero`. El `Tablero` contiene `Ficha`s. Ninguna clase necesita saber los detalles internos de otra.

---

## Clase `Ficha`

Una ficha tiene tres estados que le importan al juego:
- Su **valor** (la letra o número que forma la pareja) — nunca cambia
- Si está **visible** (boca arriba en este turno)
- Si está **encontrada** (una pareja ya la descubrió — queda boca arriba para siempre)

El valor es de solo lectura. `visible` y `encontrada` solo deben cambiar a través de métodos controlados, no por asignación directa.

In [8]:
class Ficha:
    def __init__(self, valor):
        self._valor      = valor
        self._visible    = False
        self._encontrada = False

    @property
    def valor(self):
        return self._valor

    @property
    def visible(self):
        return self._visible

    @property
    def encontrada(self):
        return self._encontrada

    def voltear(self):
        """Cambia entre boca arriba y boca abajo."""
        self._visible = not self._visible

    def marcar_pareja(self):
        """La ficha quedó encontrada: visible para siempre."""
        self._encontrada = True
        self._visible    = True

    def __str__(self):
        return self._valor if self._visible else "?"


# Prueba rápida
f = Ficha("A")
print(f"Recién creada:   {f}")       # ?
f.voltear()
print(f"Volteada:        {f}")       # A
f.voltear()
print(f"Vuelta abajo:    {f}")       # ?
f.marcar_pareja()
print(f"Encontrada:      {f}")       # A

Recién creada:   ?
Volteada:        A
Vuelta abajo:    ?
Encontrada:      A


---

## Clase `Tablero`

El tablero guarda las fichas en una grilla M×N. Su responsabilidad principal es saber imprimirse con colores:

- Fondo **gris oscuro** → ficha boca abajo
- Fondo **amarillo** → ficha seleccionada este turno
- Fondo **verde** → pareja encontrada

El tablero recibe las fichas seleccionadas como parámetro para poder pintarlas distinto — no guarda ese estado internamente porque es información del turno, no del tablero.

In [ ]:
class Tablero:
    _OCULTA      = "\033[100m"  # gris oscuro
    _SELECCIONADA = "\033[43m"  # amarillo
    _ENCONTRADA  = "\033[42m"   # verde
    _RESET       = "\033[0m"

    def __init__(self, filas, cols):
        self.filas = filas
        self.cols  = cols
        self._grid = [[None] * cols for _ in range(filas)]

    def colocar(self, ficha, fila, col):
        self._grid[fila][col] = ficha

    def ficha_en(self, fila, col):
        return self._grid[fila][col]

    def _color(self, ficha, pos, seleccionadas):
        """Decide qué color de fondo usar para una ficha."""
        if pos in seleccionadas:
            return self._SELECCIONADA
        if ficha.encontrada:
            return self._ENCONTRADA
        return self._OCULTA

    def imprimir(self, seleccionadas=None):
        seleccionadas = seleccionadas or []

        # encabezado de columnas: 1, 2, 3...
        print("\n    ", end="")
        for c in range(self.cols):
            print(f"  {c + 1} ", end="")
        print()

        # filas: A, B, C...
        for f in range(self.filas):
            print(f"  {chr(65 + f)} ", end="")
            for c in range(self.cols):
                ficha = self._grid[f][c]
                fondo = self._color(ficha, (f, c), seleccionadas)
                print(f"{fondo} {ficha} {self._RESET}", end=" ")
            print()
        print()

In [7]:
# Probamos el tablero con fichas de ejemplo
t = Tablero(2, 4)
valores = ["A", "B", "A", "B", "C", "D", "C", "D"]
i = 0
for f in range(2):
    for c in range(4):
        t.colocar(Ficha(valores[i]), f, c)
        i += 1

print("Todas ocultas:")
t.imprimir()

# Simulamos que A1 está seleccionada y B1 encontrada
t.ficha_en(0, 0).voltear()
t.ficha_en(1, 0).marcar_pareja()

print("Con selección y pareja:")
t.imprimir(seleccionadas=[(0, 0)])

Todas ocultas:

      1   2   3   4 
  A  ?   ?   ?   ?  
  B  ?   ?   ?   ?  

Con selección y pareja:

      1   2   3   4 
  A  A   ?   ?   ?  
  B  C   ?   ?   ?  



---

## Clase `Juego`

El juego es quien tiene la lógica completa: preparar el tablero con pares mezclados, pedir posiciones al jugador, evaluar si hay pareja, y detectar cuándo se terminó.

El input del jugador es de la forma `A3` — una letra para la fila y un número para la columna. `_pedir_posicion` valida que sea una posición válida, que esté dentro del tablero y que la ficha no esté ya encontrada.

In [4]:
import random
import time

class Juego:
    def __init__(self, filas=4, cols=4):
        self._tablero       = Tablero(filas, cols)
        self._turnos        = 0
        self._parejas       = 0
        self._total_parejas = (filas * cols) // 2
        self._preparar()

    def _preparar(self):
        """Crea los pares de fichas, los mezcla y los coloca en el tablero."""
        letras  = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
        valores = letras[:self._total_parejas] * 2
        random.shuffle(valores)

        for f in range(self._tablero.filas):
            for c in range(self._tablero.cols):
                self._tablero.colocar(Ficha(valores.pop()), f, c)

    def _pedir_posicion(self, numero):
        """Pide y valida una posición al jugador. Retorna (fila, col)."""
        while True:
            entrada = input(f"  Ficha {numero} (ej. A3): ").strip().upper()

            if len(entrada) < 2:
                print("  Entrada inválida.")
                continue

            fila = ord(entrada[0]) - 65
            try:
                col = int(entrada[1:]) - 1
            except ValueError:
                print("  Entrada inválida.")
                continue

            if not (0 <= fila < self._tablero.filas and 0 <= col < self._tablero.cols):
                print("  Posición fuera del tablero.")
                continue

            ficha = self._tablero.ficha_en(fila, col)

            if ficha.encontrada:
                print("  Esa pareja ya fue encontrada.")
                continue

            if ficha.visible:
                print("  Esa ficha ya está seleccionada.")
                continue

            return fila, col

    def _turno(self):
        """Ejecuta un turno completo: dos selecciones y evaluación."""
        # primera ficha
        f1, c1 = self._pedir_posicion(1)
        ficha1  = self._tablero.ficha_en(f1, c1)
        ficha1.voltear()
        self._tablero.imprimir(seleccionadas=[(f1, c1)])

        # segunda ficha
        f2, c2 = self._pedir_posicion(2)
        ficha2  = self._tablero.ficha_en(f2, c2)
        ficha2.voltear()
        self._tablero.imprimir(seleccionadas=[(f1, c1), (f2, c2)])

        self._turnos += 1

        if ficha1.valor == ficha2.valor:
            print("  ¡Pareja encontrada!\n")
            ficha1.marcar_pareja()
            ficha2.marcar_pareja()
            self._parejas += 1
        else:
            print("  No es pareja. Memorízalas...\n")
            time.sleep(2)
            ficha1.voltear()
            ficha2.voltear()

    def jugar(self):
        print("\n=== MEMORICE ===")
        print(f"Tablero {self._tablero.filas}x{self._tablero.cols} — {self._total_parejas} parejas\n")

        while self._parejas < self._total_parejas:
            self._tablero.imprimir()
            print(f"  Turno {self._turnos + 1}  |  Parejas: {self._parejas}/{self._total_parejas}\n")
            self._turno()

        self._tablero.imprimir()
        print(f"  ¡Ganaste en {self._turnos} turnos!")

---

## A jugar

In [5]:
juego = Juego(filas=4, cols=4)
juego.jugar()


=== MEMORICE ===
Tablero 4x4 — 8 parejas


      1   2   3   4 
  A  ?   ?   ?   ?  
  B  ?   ?   ?   ?  
  C  ?   ?   ?   ?  
  D  ?   ?   ?   ?  

  Turno 1  |  Parejas: 0/8


      1   2   3   4 
  A  D   ?   ?   ?  
  B  ?   ?   ?   ?  
  C  ?   ?   ?   ?  
  D  ?   ?   ?   ?  


      1   2   3   4 
  A  D   ?   ?   ?  
  B  ?   ?   C   ?  
  C  ?   ?   ?   ?  
  D  ?   ?   ?   ?  

  No es pareja. Memorízalas...


      1   2   3   4 
  A  ?   ?   ?   ?  
  B  ?   ?   ?   ?  
  C  ?   ?   ?   ?  
  D  ?   ?   ?   ?  

  Turno 2  |  Parejas: 0/8


      1   2   3   4 
  A  ?   A   ?   ?  
  B  ?   ?   ?   ?  
  C  ?   ?   ?   ?  
  D  ?   ?   ?   ?  



KeyboardInterrupt: Interrupted by user

---

## Para discutir

1. `_pedir_posicion` valida cuatro condiciones antes de aceptar una entrada. ¿Cuáles son y por qué cada una es necesaria?

2. `imprimir` recibe `seleccionadas` como parámetro en vez de guardarlo como atributo del tablero. ¿Por qué es mejor así?

3. ¿Qué diferencia hay entre `voltear()` y `marcar_pareja()`? ¿Por qué existen los dos en vez de solo uno?

---

## Ejercicio

Agrega al juego un **modo difícil**: las fichas no seleccionadas vuelven a ocultarse después de 1 segundo en vez de 2, y el tablero no se muestra entre turnos — el jugador tiene que recordar sin ayuda visual.

Implementa esto como una subclase `JuegoDificil(Juego)` que sobreescriba solo lo que cambia.

In [ ]:
# JuegoDificil
